### Load packages

In [180]:
import pandas as pd
import os
import numpy as np


### Set file information

In [181]:
# Unified VCT file info

vct_directory = r"..\data\unified_csvs"
vct_file = "vct_unified"

vct_path = os.path.join(vct_directory, vct_file + ".csv")
print(vct_path)

..\data\unified_csvs\vct_unified.csv


In [182]:
# Output file info

output_directory = r"..\data\unified_csvs"
output_file = "vct_unified_prepped"

output_path = os.path.join(output_directory, output_file + ".csv")
print(output_path)

..\data\unified_csvs\vct_unified_prepped.csv


### Read in data

In [183]:
# Read in Vermont Cyanobacteria Tracker unified csv file

vct_raw_df = pd.read_csv(vct_path)

vct_raw_df.head()

,REPORTDATE,REPORTTIME,WATERBODY,REGION,MUNICIPALITY,SITE,STATION,BLOOMINTENSITY,REPORTFREQUENCY,AFFILIATION,WEBSTATUS,DETAILS,WATERTEMP,WATERSURFACE,CYANOTAXA,OTHERTAXA,MICROCYSTIN,ANATOXIN,LATITUDE,LONGITUDE
0,08/01/2022,10:45 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,76.0,Rolling,NaN,NaN,NaN,NaN,43.71400891,-73.383001
1,08/22/2022,10:00 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,77.0,Calm,NaN,NaN,NaN,NaN,43.71400891,-73.383001
2,10/05/2022,10:30 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,57.0,Calm,NaN,NaN,NaN,NaN,43.71400891,-73.383001
3,10/21/2022,10:30 AM,Lake Champlain,Champlain - South Lake,Benson,2.0,LTM 02,1b - No Cyanobacteria Observed - brown or turb...,Routine - Biweekly,VT DEC,Generally Safe,NaN,52.0,Rolling,NaN,NaN,NaN,NaN,43.71400891,-73.383001
4,06/23/2022,2:35 PM,Lake Champlain,Champlain - Main Lake South,Panton,3.0,Arnold Bay,1a - No Cyanobacteria Observed - clear water,Routine - Weekly,LCC Volunteer,Generally Safe,NaN,NaN,Rolling,NaN,NaN,NaN,NaN,44.14952128,-73.367368


### Format data & columns

In [184]:
# Make safe copy of original data

vct_cleancols_df = vct_raw_df.copy()

In [185]:
vct_cleancols_df.columns

Index(['REPORTDATE', 'REPORTTIME', 'WATERBODY', 'REGION', 'MUNICIPALITY',
       'SITE', 'STATION', 'BLOOMINTENSITY', 'REPORTFREQUENCY', 'AFFILIATION',
       'WEBSTATUS', 'DETAILS', 'WATERTEMP', 'WATERSURFACE', 'CYANOTAXA',
       'OTHERTAXA', 'MICROCYSTIN', 'ANATOXIN', 'LATITUDE', 'LONGITUDE'],
      dtype='object')

In [186]:
# Turn all columns into lower case and then give some of them nicer names

vct_cleancols_df.columns = vct_cleancols_df.columns.str.lower()

col_map = {
    "reportdate": "report_date",
    "bloomintensity": "bloom_intensity",
    "watertemp": "water_temp",
    "watersurface": "water_surface"
}

vct_cleancols_df.rename(columns=col_map, inplace=True)

In [187]:
# Force dates to MM/DD/YYYY format

vct_cleancols_df["report_date"] = (
    pd.to_datetime(vct_cleancols_df["report_date"], errors="coerce")
      .dt.strftime("%m/%d/%Y")
)

In [188]:
# Force integers columns to integers

vct_cleancols_df["site"] = (
    pd.to_numeric(vct_cleancols_df["site"], errors="coerce")
        .astype("Int64")
)

In [189]:
# QC BLOOMINTENSITY

mask = ~vct_cleancols_df["bloom_intensity"].str.startswith(
    tuple(["1", "2", "3"]),
    na=False
)
bloom_intensity_df = vct_cleancols_df[mask]
bloom_intensity_df["bloom_intensity"].value_counts()

bloom_intensity
Tiered Alert - Quantitative    176
Quantitative                    62
Qualitative                     16
Alert 1                          7
Alert 2                          3
Vigilance                        2
Tiered Alert - Vigilance         2
Tiered Alert - Alert 1           2
Water Sample Only                1
Name: count, dtype: int64

In [190]:
# Filter for rows where BLOOMINTENSITY start with 1, 2, or 3

print("Rows before filtering:", len(vct_cleancols_df))

vct_intensity_df = vct_cleancols_df[
    vct_cleancols_df["bloom_intensity"].str.startswith(("1", "2", "3"), na=False)
]

print("Rows after filtering:", len(vct_intensity_df))

# print(vct_intensity_df["bloom_intensity"].unique())

Rows before filtering: 20601
Rows after filtering: 19604


In [191]:
# Create new column as first two characters of bloom_intensity for our targets

vct_bloom_df = vct_intensity_df.copy()

vct_bloom_df["target_bloom_intensity"] = (vct_bloom_df["bloom_intensity"].astype(str).str[:2]
                                          .str.strip()
                                          .where(vct_bloom_df["bloom_intensity"].notna())
)

first_digit = (
    vct_bloom_df["bloom_intensity"]
        .astype(str)
        .str.strip()
        .str[0]
)

vct_bloom_df["target_bloom"] = (
    first_digit.isin(["2", "3"])
)

# Set target_bloom_tf to 0/1 instead of T/F
vct_bloom_df["target_bloom"] = vct_bloom_df["target_bloom"].astype(int)

print("Bloom Intensity Code:", vct_bloom_df["target_bloom_intensity"].unique())
print("Bloom 0/1:", vct_bloom_df["target_bloom"].unique())

Bloom Intensity Code: ['1b' '1a' '1c' '1d' '2' '3']
Bloom 0/1: [0 1]


In [192]:
# Drop unnecessary columns

cols_to_drop = [
    "municipality",
    "reporttime",
    "reportfrequency",
    "affiliation",
    "details",
    "webstatus",
    "bloom_intensity",
    "microcystin", # this one was not filled in consistently (even when microcystin was listed in cyanotaxa)
    "anatoxin", # same with the other speces columns
    "othertaxa"
]

vct_dropcols_df = vct_bloom_df.drop(columns=cols_to_drop)
vct_dropcols_df.head()

,report_date,waterbody,region,site,station,water_temp,water_surface,cyanotaxa,latitude,longitude,target_bloom_intensity,target_bloom
0,08/01/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,76.0,Rolling,NaN,43.71400891,-73.383001,1b,0
1,08/22/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,77.0,Calm,NaN,43.71400891,-73.383001,1b,0
2,10/05/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,57.0,Calm,NaN,43.71400891,-73.383001,1b,0
3,10/21/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,52.0,Rolling,NaN,43.71400891,-73.383001,1b,0
4,06/23/2022,Lake Champlain,Champlain - Main Lake South,3,Arnold Bay,NaN,Rolling,NaN,44.14952128,-73.367368,1a,0


### Filter for Lake Champlain

In [193]:
# Find all Lake Champlain sites

vct_champlain_qc = vct_dropcols_df[
    vct_dropcols_df["waterbody"].str.contains("CHA", case=False, na=False)
]
vct_champlain_qc["waterbody"].unique()

array(['Lake Champlain', 'Lake Champlain ', 'Lake Champalain',
       'Lake CHamplain', 'Lake champlain', 'lake champlain',
       ' Lake Champlain '], dtype=object)

In [194]:
# Set WATERBODY to Lake Champlain for everything close to "Lake Champlain"

mask = vct_dropcols_df["waterbody"].str.contains("CHA", case=False, na=False)
vct_dropcols_df.loc[mask, "waterbody"] = "Lake Champlain"

In [195]:
# Filter for waterbody = Lake Champlain

print("Rows before filtering:", len(vct_dropcols_df))

vct_champlain_df = vct_dropcols_df[vct_dropcols_df["waterbody"] == "Lake Champlain"]

print("Rows after filtering:", len(vct_champlain_df))

print(vct_champlain_df["waterbody"].unique())

Rows before filtering: 19604
Rows after filtering: 15900
['Lake Champlain']


In [196]:
vct_champlain_df.head()

,report_date,waterbody,region,site,station,water_temp,water_surface,cyanotaxa,latitude,longitude,target_bloom_intensity,target_bloom
0,08/01/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,76.0,Rolling,NaN,43.71400891,-73.383001,1b,0
1,08/22/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,77.0,Calm,NaN,43.71400891,-73.383001,1b,0
2,10/05/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,57.0,Calm,NaN,43.71400891,-73.383001,1b,0
3,10/21/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,52.0,Rolling,NaN,43.71400891,-73.383001,1b,0
4,06/23/2022,Lake Champlain,Champlain - Main Lake South,3,Arnold Bay,NaN,Rolling,NaN,44.14952128,-73.367368,1a,0


### One-hot encoding for cyanotaxa

Note that most common cyanotaxa in Vermont are: Anabaena, Aphanizomenon, Microcystis and Oscillatoria, per this paper:

    https://www.mdpi.com/1660-4601/12/9/11560

In [197]:
vct_champlain_df["cyanotaxa"].unique()

array([nan,
       'Aphanizomenon flos-aquae;Limnotrhix;Microcystis aeruginosa;Aphanothece;Pseudanabaena;Microcystis sp. (pico)',
       'Aphanizomenon flos-aquae;Dolichospermum sp;Dolichospermum planctonicum;Limnotrhix;Microcystis aeruginosa',
       'Aphanizomenon flos-aquae;Dolichospermum crassum var. spiroides;Dolichospermum sp;Dolichospermum planctonicum;Limnotrhix',
       'none', 'Dolichospermum sp;Snowella Sp.', 'Dolichospermum sp',
       'Dolichospermum planctonicum', 'Microcystis aeruginosa',
       'Aphanizomenon flos-aquae;Dolichospermum sp;Dolichospermum crassum var. spiroides;Pseudanabaena;Limnotrhix',
       'Aphanizomenon flos-aquae;Dolichospermum sp;Dolichospermum crassum var. spiroides;Dolichospermum planctonicum;Microcystis weisenbergii;Microcystis aeruginosa',
       'Aphanizomenon flos-aquae;Dolichospermum crassum var. spiroides;Dolichospermum sp;Dolichospermum planctonicum;Microcystis aeruginosa;Microcystis sp. (pico);Microcystis weisenbergii',
       'Aphanizome

In [198]:
# Look for: Anabaena, Aphanizomenon, Microcystis and Oscillatoria

# anabaena_mask = vct_champlain_df["cyanotaxa"].str.contains("anabaena", case=False, na=False)
anabaena_df = vct_champlain_df[vct_champlain_df["cyanotaxa"].str.contains("anabaen", case=False, na=False)]
print("Anabaena records:", len(anabaena_df))

# aphanizomenon_mask = vct_champlain_df["cyanotaxa"].str.contains("aphanizomenon", case=False, na=False)
aphanizomenon_df = vct_champlain_df[vct_champlain_df["cyanotaxa"].str.contains("aphanizomen", case=False, na=False)]
print("Aphanizomenon records:", len(aphanizomenon_df))

# microcystis_mask = vct_champlain_df["cyanotaxa"].str.contains("microcystis", case=False, na=False)
microsystis_df = vct_champlain_df[vct_champlain_df["cyanotaxa"].str.contains("microcyst", case=False, na=False)]
print("Microsystin records:", len(microsystis_df))

oscillatoria_mask = vct_champlain_df["cyanotaxa"].str.contains("oscillatoria", case=False, na=False)
oscillatoria_df = vct_champlain_df[vct_champlain_df["cyanotaxa"].str.contains("oscillator", case=False, na=False)]
print("Oscillatoria records:", len(oscillatoria_df))



Anabaena records: 616
Aphanizomenon records: 553
Microsystin records: 340
Oscillatoria records: 63


In [199]:
# One-hot encoding of these four types

vct_taxa_df = vct_champlain_df.copy()

taxa_map = {
    "anabaena": "anabaen",
    "aphanizomenon": "aphanizomen",
    "microcystin": "microcyst",
    "oscillatoria": "oscillator"
}

for col_name, pattern in taxa_map.items():
    vct_taxa_df[col_name] = vct_taxa_df["cyanotaxa"].str.contains(pattern, case=False, na=False)

# Set columns to 0/1 instead of T/F
vct_taxa_df["anabaena"] = vct_taxa_df["anabaena"].astype(int)
vct_taxa_df["aphanizomenon"] = vct_taxa_df["aphanizomenon"].astype(int)
vct_taxa_df["microcystin"] = vct_taxa_df["microcystin"].astype(int)
vct_taxa_df["oscillatoria"] = vct_taxa_df["oscillatoria"].astype(int)

vct_taxa_df.head()

,report_date,waterbody,region,site,station,water_temp,water_surface,cyanotaxa,latitude,longitude,target_bloom_intensity,target_bloom,anabaena,aphanizomenon,microcystin,oscillatoria
0,08/01/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,76.0,Rolling,NaN,43.71400891,-73.383001,1b,0,0,0,0,0
1,08/22/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,77.0,Calm,NaN,43.71400891,-73.383001,1b,0,0,0,0,0
2,10/05/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,57.0,Calm,NaN,43.71400891,-73.383001,1b,0,0,0,0,0
3,10/21/2022,Lake Champlain,Champlain - South Lake,2,LTM 02,52.0,Rolling,NaN,43.71400891,-73.383001,1b,0,0,0,0,0
4,06/23/2022,Lake Champlain,Champlain - Main Lake South,3,Arnold Bay,NaN,Rolling,NaN,44.14952128,-73.367368,1a,0,0,0,0,0


In [200]:
print("anabaena value counts:\n", vct_taxa_df["anabaena"].value_counts())
print("aphanizomenon value counts:\n", vct_taxa_df["aphanizomenon"].value_counts())
print("microcystin value counts:\n", vct_taxa_df["microcystin"].value_counts())
print("oscillatoria value counts:\n", vct_taxa_df["oscillatoria"].value_counts())

anabaena value counts:
 anabaena
0    15284
1      616
Name: count, dtype: int64
aphanizomenon value counts:
 aphanizomenon
0    15347
1      553
Name: count, dtype: int64
microcystin value counts:
 microcystin
0    15560
1      340
Name: count, dtype: int64
oscillatoria value counts:
 oscillatoria
0    15837
1       63
Name: count, dtype: int64


### Reorder & rename columns for output

In [201]:
# Reorder columns

col_order = [
    "report_date",
    "waterbody",
    "region",
    "site",
    "station",
    "latitude",
    "longitude",
    "cyanotaxa",
    "water_temp",
    "water_surface",
    "anabaena",
    "aphanizomenon",
    "microcystin",
    "oscillatoria",
    "target_bloom_intensity",
    "target_bloom",
]

# Reorder the dataframe
vct_clean_df = vct_taxa_df[col_order]

In [202]:
# Rename columns

col_map = {
    "water_temp": "feature_water_temp",
    "water_surface": "feature_water_surface",
    "anabaena": "feature_anabaena",
    "aphanizomenon": "feature_aphanizomenon",
    "microcystin": "feature_microcystin",
    "oscillatoria": "feature_oscillatoria",
}

vct_clean_df.rename(columns=col_map, inplace=True)

### Calculate density of features

In [203]:
vct_clean_df.columns

Index(['report_date', 'waterbody', 'region', 'site', 'station', 'latitude',
       'longitude', 'cyanotaxa', 'feature_water_temp', 'feature_water_surface',
       'feature_anabaena', 'feature_aphanizomenon', 'feature_microcystin',
       'feature_oscillatoria', 'target_bloom_intensity', 'target_bloom'],
      dtype='object')

In [204]:
# Presence of blooms and cyanotaxa

cyano_features = ["feature_anabaena", "feature_aphanizomenon", "feature_microcystin", "feature_oscillatoria"]

# Percent of cyano values
for feature in cyano_features:
    percent_1 = (vct_clean_df[feature] == 1).mean() * 100
    print(f"{feature}: {percent_1:.2f}%")

# Percent of days with blooms
percent_bloom = (vct_clean_df["target_bloom"] == 1).mean() * 100
print(f"Bloom: {percent_bloom:.2f}%")

feature_anabaena: 3.87%
feature_aphanizomenon: 3.48%
feature_microcystin: 2.14%
feature_oscillatoria: 0.40%
Bloom: 13.37%


In [205]:
# Coverage for water features

# Get year from the date
vct_clean_df["report_date"] = pd.to_datetime(vct_clean_df["report_date"])
vct_clean_df["year"] = vct_clean_df["report_date"].dt.year

# Columns that must all be non-NaN
# required_cols = ["feature_water_temp"] + cyano_features + ["target_bloom"]
features = ["feature_water_temp", "feature_water_surface"]

for feature in features:

    yearly_percents = []
    total_valid_days = 0
    total_possible_days = 0

    for year, df_year in vct_clean_df.groupby("year"):

        min_date = df_year["report_date"].min()
        max_date = df_year["report_date"].max()

        full_range = pd.date_range(start=min_date, end=max_date, freq="D")

        # Days where this feature is non-NaN
        valid_days = (
            df_year
            .dropna(subset=[feature])
            ["report_date"]
            .dt.normalize()
            .unique()
        )

        percent_valid = len(valid_days) / len(full_range) * 100
        yearly_percents.append(percent_valid)

        total_valid_days += len(valid_days)
        total_possible_days += len(full_range)

    # Straight average across years
    avg_percent = np.mean(yearly_percents)

    # Weighted overall percent
    weighted_percent = (total_valid_days / total_possible_days) * 100

    print(f"\n{feature}")
    # print(f"Average yearly percent: {avg_percent:.2f}%")
    print(f"Weighted overall percent: {weighted_percent:.2f}%")



feature_water_temp
Weighted overall percent: 78.37%

feature_water_surface
Weighted overall percent: 88.72%


### Trailing windows for feature data

In [206]:
vct_clean_df.columns

Index(['report_date', 'waterbody', 'region', 'site', 'station', 'latitude',
       'longitude', 'cyanotaxa', 'feature_water_temp', 'feature_water_surface',
       'feature_anabaena', 'feature_aphanizomenon', 'feature_microcystin',
       'feature_oscillatoria', 'target_bloom_intensity', 'target_bloom',
       'year'],
      dtype='object')

In [207]:
# Compute trailing days for features

# Right now I'm only grouping on report date - we should talk about if we want to add in other criteria to group by
# vct_trailing_df = vct_clean_df.sort_values(["site", "report_date", "region", "station"]).copy()
vct_trailing_df = vct_clean_df.sort_values(["report_date"]).copy()
vct_trailing_df["report_date"] = pd.to_datetime(vct_trailing_df["report_date"])
vct_trailing_df = vct_trailing_df.set_index("report_date")

trailing_cat_days = 7
trailing_cont_days = str(trailing_cat_days) + "D"

trailing_cont_features = ["feature_water_temp"]
trailing_cat_features = ["feature_water_surface", "feature_anabaena", "feature_aphanizomenon", "feature_microcystin", "feature_oscillatoria"]

for cont_feature in trailing_cont_features:
    feature_name = cont_feature + "_trailing"
    vct_trailing_df[feature_name] = (
        pd.to_numeric(vct_trailing_df[cont_feature], errors="coerce")
        .shift(1)
        .rolling(trailing_cont_days, min_periods=1)
        .mean()
    )

for cat_feature in trailing_cat_features:
    feature_name = cat_feature + "_trailing"
    result = []
    values = vct_trailing_df[cat_feature].values
    for i in range(len(values)):
        start_idx = max(0, i - trailing_cat_days)
        window = values[start_idx:i]  
        window = [v for v in window if pd.notna(v)]  
        if window:
            most_common = max(set(window), key=window.count)
            result.append(most_common)
        else:
            result.append(np.nan)

    vct_trailing_df[feature_name] = result

vct_trailing_df.head()

,waterbody,region,site,station,latitude,longitude,cyanotaxa,feature_water_temp,feature_water_surface,feature_anabaena,...,feature_oscillatoria,target_bloom_intensity,target_bloom,year,feature_water_temp_trailing,feature_water_surface_trailing,feature_anabaena_trailing,feature_aphanizomenon_trailing,feature_microcystin_trailing,feature_oscillatoria_trailing
report_date,,,,,,,,,,,,,,,,,,,,,
2013-06-16,Lake Champlain,NaN,10,Donaldson Pt,44.993201,-73.175316,NaN,69.0,Rolling,0,...,0,1b,0,2013,NaN,NaN,NaN,NaN,NaN,NaN
2013-06-16,Lake Champlain,NaN,53,Beadles Cove,43.84949308,-73.370591,NaN,68.3,Rolling,0,...,0,1b,0,2013,69.00,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,81,Milton,44.65900749,-73.214242,NaN,55.0,Rolling,0,...,0,1a,0,2013,68.65,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,87,Rt. 78 Boat Access,44.97085388,-73.211652,NaN,68.3,Calm,0,...,0,1b,0,2013,64.10,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,104,Rt.2 South of N. Hero Bridge,44.87962795,-73.272149,NaN,65.8,Rolling,0,...,0,1b,0,2013,65.15,Rolling,0.0,0.0,0.0,0.0


### Output file

In [208]:
output_df = vct_trailing_df.copy()
output_df.head()

,waterbody,region,site,station,latitude,longitude,cyanotaxa,feature_water_temp,feature_water_surface,feature_anabaena,...,feature_oscillatoria,target_bloom_intensity,target_bloom,year,feature_water_temp_trailing,feature_water_surface_trailing,feature_anabaena_trailing,feature_aphanizomenon_trailing,feature_microcystin_trailing,feature_oscillatoria_trailing
report_date,,,,,,,,,,,,,,,,,,,,,
2013-06-16,Lake Champlain,NaN,10,Donaldson Pt,44.993201,-73.175316,NaN,69.0,Rolling,0,...,0,1b,0,2013,NaN,NaN,NaN,NaN,NaN,NaN
2013-06-16,Lake Champlain,NaN,53,Beadles Cove,43.84949308,-73.370591,NaN,68.3,Rolling,0,...,0,1b,0,2013,69.00,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,81,Milton,44.65900749,-73.214242,NaN,55.0,Rolling,0,...,0,1a,0,2013,68.65,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,87,Rt. 78 Boat Access,44.97085388,-73.211652,NaN,68.3,Calm,0,...,0,1b,0,2013,64.10,Rolling,0.0,0.0,0.0,0.0
2013-06-16,Lake Champlain,NaN,104,Rt.2 South of N. Hero Bridge,44.87962795,-73.272149,NaN,65.8,Rolling,0,...,0,1b,0,2013,65.15,Rolling,0.0,0.0,0.0,0.0


### Merge into one unified csv

In [209]:
# Output the unified csv file

output_df.to_csv(output_path, index=False)